In [24]:
# Imports
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os



In [25]:
DATA_PATH = "FareedData_End.xlsx"
df = pd.read_excel(DATA_PATH)
# Normalize column names and use standard names: ID, ProjectTitle, Abstract
df.columns = df.columns.str.strip()
if df.shape[1] >= 3:
    df = df.iloc[:, :3].copy()
    df.columns = ['ID', 'ProjectTitle', 'Abstract']
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [26]:
df.head(10)

,ID,ProjectTitle,Abstract
0,1,Creation of the task builder App to Assist peo...,"An overview of the Task Builder app, a task-ma..."
1,2,Bilingual Buddy\n,Bilingual students often face challenges in ma...
2,3,Documentation-Aware Code Generation Via RAG\n,Modern large language models (LLMs) increase t...
3,4,Vintage Game Emulator,This project explores the integration of artif...
4,5,An AI-Driven Microfinance Platform to Enhance ...,Microfinance provides financial services to bu...
5,6,AR Storybook,Parents often use technology as a digital paci...
6,7,IoTsolate: Network Microsegmentation for Manag...,Developing solutions to secure IoT devices is ...
7,8,Personal Trip Planner,Generating routes with multiple destinations c...
8,9,AI-Powered IoT Monitoring and Security for Sma...,Smart home devices are becoming increasingly p...
9,10,EMT Vision,Augmented Reality (AR) has demonstrated consid...


In [27]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (427, 3)


In [28]:
print("Number of projects:", len(df))

Number of projects: 427


In [29]:
print("\nColumn names:", list(df.columns))


Column names: ['ID', 'ProjectTitle', 'Abstract']


In [30]:
print("\nData types:\n", df.dtypes)


Data types:
 ID               int64
ProjectTitle    object
Abstract        object
dtype: object


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 427 entries, 0 to 426
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ID            427 non-null    int64 
 1   ProjectTitle  427 non-null    object
 2   Abstract      398 non-null    object
dtypes: int64(1), object(2)
memory usage: 10.1+ KB


In [32]:
def preprocess_text(text):
    """Clean and normalize a single text string."""
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = str(text).lower().strip()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Optional: keep letters, numbers, and spaces (remove special chars that add noise)
    # text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # text = re.sub(r'\s+', ' ', text).strip()
    return text

# Fill missing abstracts with empty string and preprocess
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['ProjectTitle_clean'] = df['ProjectTitle'].apply(preprocess_text)
# Combined text: title + abstract (so similarity uses both)
df['Combined_Text'] = (df['ProjectTitle_clean'] + ' ' + df['Abstract_clean']).str.strip()


In [33]:
# Drop rows where combined text is empty (no title and no abstract)
df = df[df['Combined_Text'] != ''].copy()
df = df.reset_index(drop=True)
print("Preprocessing done.")


Preprocessing done.


In [34]:
df.head()

,ID,ProjectTitle,Abstract,Abstract_clean,ProjectTitle_clean,Combined_Text
0,1,Creation of the task builder App to Assist peo...,"An overview of the Task Builder app, a task-ma...","an overview of the task builder app, a task-ma...",creation of the task builder app to assist peo...,creation of the task builder app to assist peo...
1,2,Bilingual Buddy\n,Bilingual students often face challenges in ma...,bilingual students often face challenges in ma...,bilingual buddy,bilingual buddy bilingual students often face ...
2,3,Documentation-Aware Code Generation Via RAG\n,Modern large language models (LLMs) increase t...,modern large language models (llms) increase t...,documentation-aware code generation via rag,documentation-aware code generation via rag mo...
3,4,Vintage Game Emulator,This project explores the integration of artif...,this project explores the integration of artif...,vintage game emulator,vintage game emulator this project explores th...
4,5,An AI-Driven Microfinance Platform to Enhance ...,Microfinance provides financial services to bu...,microfinance provides financial services to bu...,an ai-driven microfinance platform to enhance ...,an ai-driven microfinance platform to enhance ...


In [35]:
# TF-IDF parameters (tune as needed)
MAX_FEATURES = 10000
MIN_DF = 1
MAX_DF = 0.95
NGRAM_RANGE = (1, 3)  # unigrams and bigrams for better semantic capture
STOP_WORDS = 'english'

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    max_df=MAX_DF,
    ngram_range=NGRAM_RANGE,
    stop_words=STOP_WORDS,
    strip_accents='unicode',
    lowercase=True
)
corpus = df['Combined_Text'].tolist()
tfidf_matrix = vectorizer.fit_transform(corpus)

print("Vocabulary size (max features used):", len(vectorizer.get_feature_names_out()))
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Vocabulary size (max features used): 10000
TF-IDF matrix shape: (427, 10000)


In [36]:
MODEL_DIR = "fareed_model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save vectorizer
with open(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)

# Save TF-IDF matrix
with open(os.path.join(MODEL_DIR, "tfidf_matrix.pkl"), "wb") as f:
    pickle.dump(tfidf_matrix, f)

# Save project data needed for retrieval (ID, ProjectTitle, Abstract, Combined_Text)
df_export = df[['ID', 'ProjectTitle', 'Abstract', 'Combined_Text']].copy()
df_export.to_pickle(os.path.join(MODEL_DIR, "projects_df.pkl"))

# Also save as CSV for readability
df_export.to_csv(os.path.join(MODEL_DIR, "projects_metadata.csv"), index=False)

print("Model and data saved in folder:", MODEL_DIR)


Model and data saved in folder: fareed_model


We performed a self-similarity validation test where each project document is compared with itself using cosine similarity. The similarity value should equal 1.0, which confirms that the TF-IDF vectorization and similarity computation are functioning correctly.

In [37]:
# 1) Self-similarity: diagonal of cosine_similarity(tfidf_matrix, tfidf_matrix) should be 1.0
sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
self_sim = np.diag(sim_matrix)
print("Self-similarity (min, max, mean):", self_sim.min(), self_sim.max(), self_sim.mean())
assert np.allclose(self_sim, 1.0), "Self-similarity should be 1.0"
print("Self-similarity check passed.")

Self-similarity (min, max, mean): 0.9999999999999879 1.0000000000000149 1.0
Self-similarity check passed.


In [38]:
def check_project_simple(new_title, new_abstract, similarity_threshold=0.6):

    # 1. Preprocess the input
    text = preprocess_text(new_title) + " " + preprocess_text(new_abstract)

    # 2. Convert to TF-IDF vector
    new_vector = vectorizer.transform([text])

    # 3. Compute similarity with all projects
    similarities = cosine_similarity(new_vector, tfidf_matrix).flatten()

    # 4. Find highest similarity
    max_similarity = similarities.max()
    max_index = similarities.argmax()

    similarity_percent = max_similarity * 100

    print("\nInput Title:", new_title)
    print("Similarity Score: %.2f%%" % similarity_percent)

    # 5. Classification
    if max_similarity >= similarity_threshold:
        print("Result: DUPLICATE")

        print("\nMost Similar Project:")
        print("Project ID:", df.iloc[max_index]['ID'])
        print("Title:", df.iloc[max_index]['ProjectTitle'])

        if pd.notna(df.iloc[max_index]['Abstract']):
            print("Abstract:", df.iloc[max_index]['Abstract'])

    else:
        print("Result: UNIQUE")

In [39]:
new_title = "Istaqim An Assistant Application to Correct Prayer for Arab Muslims"

new_abstract = "Prayer is the second pillar of Islam- a link between the servant and his Lord- and Muslims must perform it five times a day. There are many postures in the pillars of prayer and its duties that must be performed in a precise manner. However- many Muslims- young and old- do not perform prayer properly due to having learned to pray incorrectly- having no one to personally guide them- or being new to prayer. To address this issue- we proposed developing an Artificial Intelligence as- sistant app using deep learning to guide worshipers by detecting the wrong postures in their prayers- assessing their mistakes- and showing corrections. The Istaqim application came to achieve this goal by training the YOLOv5 neural network to recognize the correct prayer postures. The results are shown with pictures and the percentage of the error in each prayer posture."

threshold = 0.25

check_project_simple(new_title, new_abstract, threshold)


Input Title: Istaqim An Assistant Application to Correct Prayer for Arab Muslims
Similarity Score: 100.00%
Result: DUPLICATE

Most Similar Project:
Project ID: 100
Title: Istaqim An Assistant Application to Correct Prayer for Arab Muslims
Abstract: Prayer is the second pillar of Islam- a link between the servant and his Lord- and Muslims must perform it five times a day. There are many postures in the pillars of prayer and its duties that must be performed in a precise manner. However- many Muslims- young and old- do not perform prayer properly due to having learned to pray incorrectly- having no one to personally guide them- or being new to prayer. To address this issue- we proposed developing an Artificial Intelligence as- sistant app using deep learning to guide worshipers by detecting the wrong postures in their prayers- assessing their mistakes- and showing corrections. The Istaqim application came to achieve this goal by training the YOLOv5 neural network to recognize the correc

In [40]:
new_title = "Context-Aware Smart Attendance System Using Multi-Sensor Fusion and Engagement Analysis"

new_abstract = "This project proposes a context-aware attendance system that utilizes multi-sensor data fusion instead of relying solely on facial recognition. The system combines environmental signals such as device proximity (Bluetooth/Wi-Fi), classroom activity patterns, and optional lightweight identity verification to confirm student presence. Additionally, it incorporates engagement analysis by monitoring interaction levels (e.g., participation, device usage patterns) to differentiate between mere presence and actual involvement. The system applies intelligent models to detect inconsistencies such as proxy attendance or abnormal participation behavior. This approach enhances reliability, reduces dependency on a single biometric method, and introduces a more holistic understanding of student attendance and engagement."

threshold = 0.25

check_project_simple(new_title, new_abstract, threshold)


Input Title: Context-Aware Smart Attendance System Using Multi-Sensor Fusion and Engagement Analysis
Similarity Score: 23.23%
Result: UNIQUE
